# Project 93 — Local PR Review Assistant
## Diff Analysis → Code Review → Risk Assessment

**Stack:** LangChain · Ollama · Pydantic · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama pydantic pandas

## Step 1 — Simulated PR Diffs

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
import json, pandas as pd

llm = ChatOllama(model="qwen3:8b", temperature=0.1)

pull_requests = [
    {
        "id": "PR-142",
        "title": "Add rate limiting to API endpoints",
        "author": "alice",
        "files_changed": 4,
        "diff": """
--- a/api/middleware.py
+++ b/api/middleware.py
@@ -1,5 +1,28 @@
+from time import time
+from collections import defaultdict
+
+class RateLimiter:
+    def __init__(self, max_requests=100, window=60):
+        self.max_requests = max_requests
+        self.window = window
+        self.requests = defaultdict(list)
+
+    def is_allowed(self, client_ip):
+        now = time()
+        self.requests[client_ip] = [
+            t for t in self.requests[client_ip] if now - t < self.window
+        ]
+        if len(self.requests[client_ip]) >= self.max_requests:
+            return False
+        self.requests[client_ip].append(now)
+        return True
+
 from flask import Flask, request, jsonify
+
+limiter = RateLimiter()
+
 @app.before_request
 def check_rate_limit():
+    if not limiter.is_allowed(request.remote_addr):
+        return jsonify({"error": "rate limited"}), 429
"""
    },
    {
        "id": "PR-143",
        "title": "Fix SQL injection in user search",
        "author": "bob",
        "files_changed": 2,
        "diff": """
--- a/api/users.py
+++ b/api/users.py
@@ -10,7 +10,8 @@
 def search_users(query):
-    sql = f"SELECT * FROM users WHERE name LIKE '%{query}%'"
-    results = db.execute(sql)
+    sql = "SELECT * FROM users WHERE name LIKE :query"
+    results = db.execute(sql, {"query": f"%{query}%"})
     return results
"""
    },
    {
        "id": "PR-144",
        "title": "Add user preferences table",
        "author": "carol",
        "files_changed": 3,
        "diff": """
--- a/models/user.py
+++ b/models/user.py
@@ -15,6 +15,20 @@
+class UserPreference(Base):
+    __tablename__ = 'user_preferences'
+    id = Column(Integer, primary_key=True)
+    user_id = Column(Integer, ForeignKey('users.id'))
+    key = Column(String(100))
+    value = Column(Text)
+    created_at = Column(DateTime, default=datetime.utcnow)
+
--- /dev/null
+++ b/migrations/add_preferences.py
+def upgrade():
+    op.create_table('user_preferences',
+        sa.Column('id', sa.Integer, primary_key=True),
+        sa.Column('user_id', sa.Integer, sa.ForeignKey('users.id')),
+        sa.Column('key', sa.String(100)),
+        sa.Column('value', sa.Text),
+    )
"""
    },
]
print(f"Pull requests to review: {len(pull_requests)}")

## Step 2 — Automated Code Review

In [ ]:
class ReviewComment(BaseModel):
    file: str
    line_ref: str
    severity: str = Field(description="critical, warning, suggestion, praise")
    comment: str
    suggested_fix: str = ""

class PRReview(BaseModel):
    pr_id: str
    approval: str = Field(description="approve, request_changes, comment")
    risk_level: str = Field(description="low, medium, high, critical")
    summary: str
    comments: list[ReviewComment]
    security_concerns: list[str]
    test_coverage_needed: list[str]
    breaking_changes: bool

reviewer = llm.with_structured_output(PRReview)

reviews = []
for pr in pull_requests:
    review = reviewer.invoke(
        f"Review this pull request:\n"
        f"Title: {pr['title']}\nAuthor: {pr['author']}\n"
        f"Files changed: {pr['files_changed']}\n\nDiff:\n{pr['diff']}"
    )
    reviews.append(review)
    print(f"\n{'='*50}")
    print(f"{pr['id']}: {pr['title']}")
    print(f"  Decision: {review.approval.upper()} | Risk: {review.risk_level}")
    print(f"  Comments: {len(review.comments)}")
    for c in review.comments:
        icon = {"critical": "🔴", "warning": "🟡", "suggestion": "🔵", "praise": "🟢"}.get(c.severity, "⚪")
        print(f"    {icon} [{c.severity}] {c.comment[:70]}")
    if review.security_concerns:
        print(f"  Security: {review.security_concerns}")

## Step 3 — Risk Dashboard

In [ ]:
risk_rows = []
for pr, review in zip(pull_requests, reviews):
    risk_rows.append({
        "PR": pr["id"],
        "Title": pr["title"][:30],
        "Author": pr["author"],
        "Risk": review.risk_level,
        "Decision": review.approval,
        "Comments": len(review.comments),
        "Security": len(review.security_concerns),
        "Breaking": review.breaking_changes,
    })

df = pd.DataFrame(risk_rows)
print("PR REVIEW DASHBOARD")
print("=" * 60)
print(df.to_string(index=False))

print(f"\nSummary:")
print(f"  Approved: {sum(1 for r in reviews if r.approval == 'approve')}")
print(f"  Changes requested: {sum(1 for r in reviews if r.approval == 'request_changes')}")
print(f"  Critical risks: {sum(1 for r in reviews if r.risk_level == 'critical')}")

## Step 4 — Review Summary for Team

In [ ]:
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "Write a team-facing summary of today's PR reviews. "
     "Highlight security fixes, breaking changes, and items needing attention."),
    ("human", "Reviews:\n{reviews}")
])
summary_chain = summary_prompt | llm | StrOutputParser()

team_summary = summary_chain.invoke({
    "reviews": json.dumps([r.model_dump() for r in reviews], indent=2)
})
print("TEAM SUMMARY")
print("=" * 50)
print(team_summary[:500])

## What You Learned
- **Automated PR review** with structured feedback
- **Security concern detection** in code changes
- **Risk-level assessment** per change
- **Review dashboard** for team visibility